In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,when,count
import seaborn as sns
import matplotlib.pyplot as plt

In [0]:
spark = SparkSession.builder \
    .appName("Credit Default Prediction") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "10g") \
    .getOrCreate()

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("dbfs:/FileStore/shared_uploads/s6504053630171@email.kmutnb.ac.th/application_data-4.csv")

**วิเคาระห์ข้อมูลเบื้องต้น**

In [0]:
df.createOrReplaceTempView("mytable")
result = spark.sql("SELECT * FROM mytable")
display(result)

In [0]:
total_row = df.count()
print(f"จำนวนข้อมูลทั้งหมด : {total_row}")

In [0]:
df.printSchema()

In [0]:
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()

In [0]:
# นับจำนวนตัวอย่างในแต่ละ class
class_counts = df.groupBy("TARGET").count()

# แสดงผล
class_counts.show()

# คำนวณสัดส่วน
total_count = df.count()
class_distribution = class_counts.withColumn("percentage", (class_counts["count"] / total_count) * 100)

# แสดงผลสัดส่วน
class_distribution.show()


In [0]:
# แปลงเป็น Pandas DataFrame
class_distribution_pd = class_distribution.toPandas()

plt.figure(figsize=(8, 5))
plt.bar(class_distribution_pd['TARGET'], class_distribution_pd['percentage'], color=['blue', 'orange'])
plt.xlabel('Class (TARGET)')
plt.ylabel('Percentage')
plt.title('Class Distribution')
plt.xticks([0, 1], ['Not Default', 'Default'])
plt.show()


**Missing Value**


In [0]:
df_cleaned = df.dropna()
display(df_cleaned)


In [0]:
# สร้าง amount_category
df = df.withColumn("AMT_CATEGORY",
    when(col("AMT_INCOME_TOTAL") < 30000, "Low")
    .when((col("AMT_INCOME_TOTAL") >= 100000) & (col("AMT_INCOME_TOTAL") < 300000), "Medium")
    .otherwise("High")
)
display(df)

In [0]:
df = df[['TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'AMT_INCOME_TOTAL','AMT_CATEGORY', 
             'AMT_CREDIT', 'NAME_EDUCATION_TYPE', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 
             'REGION_RATING_CLIENT', 'REG_REGION_NOT_WORK_REGION', 'DAYS_LAST_PHONE_CHANGE']]
display(df)

In [0]:
df = df.fillna("unknow")
display(df)

**Feature Engineering**

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder

In [0]:
categorical_ohe = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR']

categorical_ohe_transformer = Pipeline(steps=[
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ])


In [0]:
categorical_oe = ['NAME_EDUCATION_TYPE', 'OCCUPATION_TYPE']

categorical_oe_transformer = Pipeline(steps=[
    ('ohe', OrdinalEncoder())
    ])

In [0]:
numerical = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'DAYS_LAST_PHONE_CHANGE']

numerical_ss = Pipeline(steps=[
    ('ss', StandardScaler())
])

In [0]:
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical_ohe_transformer', categorical_ohe_transformer, categorical_ohe),
        ('categorical_oe_transformer', categorical_oe_transformer, categorical_oe),
        ('numerical_ss', numerical_ss, numerical)
])

**การเตรียม Feature Vector**

In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

# จัดการข้อมูล Missing
df = df.fillna(0)

# แปลงข้อมูลเชิงหมวดหมู่
indexers = [
    StringIndexer(inputCol='NAME_CONTRACT_TYPE', outputCol='NAME_CONTRACT_TYPE_INDEX'),
    StringIndexer(inputCol='CODE_GENDER', outputCol='CODE_GENDER_INDEX'),
    StringIndexer(inputCol='FLAG_OWN_CAR', outputCol='FLAG_OWN_CAR_INDEX')
]

for indexer in indexers:
    df = indexer.fit(df).transform(df)

# รวมคุณสมบัติเป็น Feature Vector
assembler = VectorAssembler(inputCols=[
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 
    'DAYS_LAST_PHONE_CHANGE', 'NAME_CONTRACT_TYPE_INDEX', 'CODE_GENDER_INDEX', 
    'FLAG_OWN_CAR_INDEX'], outputCol='features')

df_vector = assembler.transform(df)

# สเกลค่า
scaler = StandardScaler(inputCol='features', outputCol='scaled_features')
df_scaled = scaler.fit(df_vector).transform(df_vector)

# เตรียมข้อมูลสุดท้ายสำหรับโมเดล
final_data = df_scaled.select('scaled_features', 'TARGET')


In [0]:
# แสดง Schema ของแต่ละ DataFrame
df_vector.printSchema()
df_scaled.printSchema()
final_data.printSchema()


**Training and Testing**

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [0]:
X = df.drop("TARGET",)
y = df["TARGET"]

In [0]:
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

In [0]:
# ตรวจสอบ Schema ของชุดข้อมูลที่แบ่งแล้ว
train_data.printSchema()
test_data.printSchema()

# แสดงข้อมูลบางส่วนเพื่อยืนยันว่าแบ่งข้อมูลสำเร็จ
train_data.show(5)
test_data.show(5)


In [0]:
print(f"Train Data Count: {train_data.count()}")
print(f"Test Data Count: {test_data.count()}")


**การสร้าง Model**

In [0]:
# Import classifiers ที่มีใน PySpark
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier,
    MultilayerPerceptronClassifier,
    NaiveBayes
)

In [0]:
# Import classifiers ที่มีใน PySpark
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier,
    MultilayerPerceptronClassifier,
    NaiveBayes
)

# 1. Logistic Regression
lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=10
)

# 2. Decision Tree
dt = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5
)

# 3. Random Forest
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=10,
    maxDepth=5
)

# 4. Gradient Boosted Trees
gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxIter=10
)

# 5. Neural Network (Multilayer Perceptron)
# ตัวอย่างการกำหนดค่า features_size
features_size = len(final_data.columns) - 1  # ลบ 1 สำหรับคอลัมน์เป้าหมาย (TARGET)
layers = [features_size, 5, 4, 2]  # ต้องกำหนดขนาดของ layers
mlp = MultilayerPerceptronClassifier(
    labelCol="label",
    featuresCol="features",
    layers=layers,
    maxIter=100
)

# 6. Naive Bayes
nb = NaiveBayes(
    labelCol="label",
    featuresCol="features"
)

# ตัวอย่างการใช้งาน Random Forest
def train_random_forest(train_data, test_data):
    # สร้างโมเดล
    rf = RandomForestClassifier(
        labelCol="label",
        featuresCol="features",
        numTrees=100,
        maxDepth=5,
        seed=42
    )
    
    # เทรนโมเดล
    model = rf.fit(train_data)
    
    # ทำนายผล
    predictions = model.transform(test_data)
    
    # ประเมินผล
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy"
    )
    
    accuracy = evaluator.evaluate(predictions)
    print(f"Accuracy: {accuracy}")
    
    return model, predictions

Logtistic Regression

In [0]:
# สร้างโมเดล Logistic Regression
lr = LogisticRegression(featuresCol='scaled_features', labelCol='TARGET')

# เทรนโมเดล
lr_model = lr.fit(train_data)

In [0]:
test_results = lr_model.transform(test_data)


In [0]:
evaluator = BinaryClassificationEvaluator(labelCol='TARGET', rawPredictionCol='prediction')
auc = evaluator.evaluate(test_results)
print(f'AUC: {auc}')


In [0]:
print(lr_model.summary)

Random Forest Classifier

In [0]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier


In [0]:
# สร้าง Random Forest Classifier model
rf = RandomForestClassifier(featuresCol='scaled_features', labelCol='TARGET', numTrees=100)

In [0]:
rf_model = rf.fit(train_data)

In [0]:
evaluator = BinaryClassificationEvaluator(labelCol='TARGET', rawPredictionCol='prediction')
auc = evaluator.evaluate(test_results)
print(f'AUC: {auc}')

In [0]:
# แสดงข้อมูล feature importances
feature_importances = rf_model.featureImportances
print(f'Feature Importances: {feature_importances}')

Decision Tree Classifier

In [0]:
from pyspark.ml.classification import DecisionTreeClassifier

In [0]:
# สร้าง Decision Tree Classifier model
dt = DecisionTreeClassifier(featuresCol='scaled_features', labelCol='TARGET', maxDepth=5)

In [0]:
dt_model = dt.fit(train_data)

In [0]:
test_results = dt_model.transform(test_data)

In [0]:
evaluator = BinaryClassificationEvaluator(labelCol='TARGET', rawPredictionCol='prediction')
auc = evaluator.evaluate(test_results)
print(f'AUC: {auc}')

In [0]:
print(dt_model.toDebugString)

XGBoostClassifier

In [0]:
pip install xgboost

In [0]:
import xgboost as xgb
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [0]:
# สมมุติว่า final_data คือ DataFrame ที่คุณต้องการใช้
feature_columns = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT']
assembler = VectorAssembler(inputCols=feature_columns, outputCol='features')
final_data = assembler.transform(df)

# แบ่งข้อมูลเป็นชุดฝึกและชุดทดสอบ
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)


In [0]:
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob', 
    num_class=3,  # ปรับจำนวนคลาสตามจำนวนคลาสในข้อมูลของคุณ
    eval_metric='mlogloss'
)


In [0]:
# แปลงข้อมูลฝึกและทดสอบเป็น DMatrix
dtrain = xgb.DMatrix(train_data.select('features').rdd.map(lambda row: row[0]).collect(), label=train_data.select('TARGET').rdd.map(lambda row: row[0]).collect())
dtest = xgb.DMatrix(test_data.select('features').rdd.map(lambda row: row[0]).collect(), label=test_data.select('TARGET').rdd.map(lambda row: row[0]).collect())


In [0]:
X = df.drop('TARGET')
y = df['TARGET']

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
xgb_model.fit(
    X_train,  # features สำหรับ training
    y_train,  # target variable
    eval_set=[(X_train, y_train), (X_test, y_test)],  # ใช้สำหรับติดตามการ train
    eval_metric='auc',  # เมทริกที่ใช้ประเมิน
    verbose=True  # แสดงผลระหว่าง training
)

In [0]:
xgb_model.fit(dtrain)

In [0]:
preds = xgb_model.predict(dtest)

In [0]:
evaluator = MulticlassClassificationEvaluator(labelCol='TARGET', predictionCol='prediction', metricName='accuracy')
accuracy = evaluator.evaluate(test_results)
print(f'Accuracy: {accuracy}')


In [0]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml import Pipeline

In [0]:

# 1. เตรียมข้อมูล
def prepare_data(df):
    # แปลง string columns เป็น numeric indices
    label_indexer = StringIndexer(
        inputCol="label_column",  # ชื่อคอลัมน์ที่เป็น label
        outputCol="label"
    )
    
    # รวม features เข้าด้วยกันเป็น vector
    feature_cols = ["feature1", "feature2", "feature3"]  # ระบุชื่อคอลัมน์ที่เป็น features
    assembler = VectorAssembler(
        inputCols=feature_cols,
        outputCol="features"
    )
    
    return label_indexer, assembler

# 2. สร้างโมเดล
def create_gbt_model():
    gbt = GBTClassifier(
        labelCol="label",
        featuresCol="features",
        maxIter=10,        # จำนวนรอบการ train สูงสุด
        maxDepth=5,        # ความลึกสูงสุดของแต่ละ tree
        stepSize=0.1       # learning rate
    )
    return gbt

# 3. สร้าง Pipeline
def create_pipeline(label_indexer, assembler, gbt):
    pipeline = Pipeline(stages=[
        label_indexer,
        assembler,
        gbt
    ])
    return pipeline

# 4. Train และ evaluate โมเดล
def train_and_evaluate(pipeline, train_data, test_data):
    # Train โมเดล
    model = pipeline.fit(train_data)
    
    # ทำนายผลบน test set
    predictions = model.transform(test_data)
    
    # ประเมินผล
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy"
    )
    accuracy = evaluator.evaluate(predictions)
    
    return model, predictions, accuracy

# การใช้งานทั้งหมด
def main(data):
    # แบ่งข้อมูล
    train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)
    
    # เตรียมข้อมูล
    label_indexer, assembler = prepare_data(data)
    
    # สร้างโมเดล
    gbt = create_gbt_model()
    
    # สร้าง pipeline
    pipeline = create_pipeline(label_indexer, assembler, gbt)
    
    # Train และ evaluate
    model, predictions, accuracy = train_and_evaluate(
        pipeline, 
        train_data, 
        test_data
    )
    
    print(f"Accuracy: {accuracy}")
    
    # แสดงตัวอย่างผลการทำนาย
    predictions.select("label", "prediction", "probability").show(5)
    
    return model, predictions

# ตัวอย่างการใช้งาน
# สมมติว่ามี DataFrame ชื่อ 'data' อยู่แล้ว
model, predictions = main(df)